In [1]:
# IN01: AI Architecture Decision Framework

# Objectives
# By the end of this notebook you will be able to:
# - Apply a five-axis decision model to any AI architecture problem
# - Distinguish between agent, workflow, and traditional software approaches with quantified rationale
# - Produce a one-page decision matrix for a given Walmart business use case
# - Validate and export your decision matrix in a format suitable for ARB submission

# Deliverable: decision_matrix.json

In [2]:
# The three architecture options
# 1. Traditional software
# 2. Workflow or chain
# 3. AI agent

In [3]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv('OPENAI_API_KEY'))
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print('Environment loaded.')
print(f'OpenAI client ready: {client is not None}')

Environment loaded.
OpenAI client ready: True


In [4]:
# Should this AI use case be implemented as an agent?

# Suppose a problem can be solved using ten fixed if-else rules. Would you build an AI agent for it?

# {
#   "purchase_age_days": 12,
#   "product_category": "electronics",
#   "receipt_available": true
# }

# Traditional software may be suitable for:
# - Calculating discounts
# - Validating coupon expiry
# - Checking inventory thresholds
# - Computing tax
# - Validating employee IDs
# - Checking whether an order qualifies for free delivery
# - Applying a clearly defined return policy

# Suppose a customer writes:
# ‘The blender started making a burning smell after two uses, but I no longer have the original box.’

# Would calculating the final bill amount require an agent?

# Workflow:
# Suppose Walmart wants to generate a daily store performance report.
# Steps:
# 1. Fetch sales data
# 2. Fetch inventory data
# 3. Calculate KPIs
# 4. Identify underperforming categories
# 5. Ask an LLM to summarize the findings
# 6. Email the report

# Suppose a supplier risk system first reads a news report.
# After reading it, the system may need to decide:
# - If the article mentions bankruptcy, inspect financial statements.
# - If it mentions a strike, check logistics impact.
# - If it mentions sanctions, consult compliance systems.
# - If no major issue is found, inspect internal delivery performance.
# The next action depends on what was discovered in the previous step.

# Suppose our process always performs these steps: load data, clean data, calculate KPIs and generate an LLM summary. Is this an agent?

# Agent:
# Open-ended reasoning: Investigate why sales of a product category suddenly declined.

# Dynamic tool selection:
# For one problem, the agent may use inventory and pricing tools.
# For another problem, it may use reviews and supplier data.
# The tool path is not fixed in advance.

# Multi-step planning under uncertainty:
# For example:
# 1. Analyse regional sales.
# 2. Find that only one region is affected.
# 3. Check inventory in that region.
# 4. Discover that inventory is normal.
# 5. Check competitor pricing.
# 6. Discover a large competitor discount.
# 7. Estimate the possible revenue impact.
# 8. Recommend a response.

# We did not know the complete route before the investigation began.

## The Architecture Decision Problem

Every AI project begins with a question the team usually answers by gut feel: *should this be an agent?*
In production, gut feel costs money. A mis-classified problem -- one that gets an agent when a workflow
was sufficient -- adds latency, cost, observability complexity, and failure surface area for no benefit.

There are three architecture archetypes:

| Archetype | When to use | When NOT to use |
|-----------|-------------|-----------------|
| **Traditional software** | Deterministic rules, structured input/output, no ambiguity | Tasks requiring language understanding or dynamic reasoning |
| **Workflow (chain)** | Known steps, fixed sequence, no branching on content | Tasks requiring mid-process decisions based on intermediate results |
| **Agent** | Open-ended reasoning, dynamic tool selection, multi-step planning under uncertainty | Latency-critical tasks, fully deterministic tasks, tasks where all steps are known upfront |

The decision is not about capability -- an agent can technically do anything a workflow can.
The decision is about cost, latency, predictability, and operational risk.

## The Five-Axis Decision Model

Score each axis 1-5 for your use case. Higher scores push toward agents.

| Axis | Score 1 | Score 5 |
|------|---------|---------|
| **Task Complexity** | Single step, fully deterministic | Multi-step, dynamic branching required |
| **Latency Requirement** | Real-time (< 500ms) | Batch or async acceptable (> 10s) |
| **Cost Ceiling** | Pennies per query | Dollars per query acceptable |
| **Risk Tolerance** | Zero hallucination tolerance | Some imprecision acceptable if caught downstream |
| **Update Frequency** | Logic changes rarely | Requirements change weekly |

In [5]:
# A score of 1 generally pushes us toward traditional software.

# A score of 5 generally pushes us toward an agent.

# Latency: A low score means the business requires a very fast response.

# A high score means the business can wait longer.

# Axis 3: Cost ceiling
# Score 1: Pennies per query
# Score 5: Dollars per query acceptable

# Which use case can tolerate a higher cost per query: checking a product price or investigating a major supplier disruption?

# Axis 4: Risk tolerance
# This measures how much uncertainty the business can tolerate.

# Axis 5: Update frequency

In [6]:
# If a use case receives these scores: 4, 4, 3, 3, and 4, which architecture will be recommended?
# 4 + 4 + 3 + 3 + 4 = 18 => Workflow

# scores = {
#     'task_complexity': 4,
#     'latency_tolerance': 3,
#     'cost_ceiling': 2,
#     'risk_tolerance': 3,
#     'update_frequency': 4
# }

In [7]:
AXES = ['task_complexity', 'latency_tolerance', 'cost_ceiling', 'risk_tolerance', 'update_frequency']

AXIS_DESCRIPTIONS = {
    'task_complexity':    'How many distinct decision points exist in the task?',
    'latency_tolerance':  'How much time can the system take to respond?',
    'cost_ceiling':       'What is the acceptable cost per query?',
    'risk_tolerance':     'How much imprecision is acceptable in the output?',
    'update_frequency':   'How often do requirements or logic change?',
}

THRESHOLDS = {
    'traditional': (5, 12),   # total score range
    'workflow':    (13, 18),
    'agent':       (19, 25),
}

def recommend_architecture(scores: dict) -> dict:
    total = sum(scores.values())
    if total <= 12:
        arch = 'Traditional Software'
        rationale = ('Low complexity, strict latency, or low cost ceiling. '
                     'A rule-based or deterministic pipeline is faster, cheaper, and more predictable.')
    elif total <= 18:
        arch = 'Workflow (Chain)'
        rationale = ('Moderate complexity with known steps. '
                     'A fixed-sequence workflow gives you LLM capability without agent overhead.')
    else:
        arch = 'Agent'
        rationale = ('High complexity, dynamic branching, or rapidly changing requirements. '
                     'An agent is justified -- but implement observability and cost controls from day one.')
    return {'architecture': arch, 'total_score': total, 'rationale': rationale}

print('Decision model loaded.')
print('Axes:', AXES)

Decision model loaded.
Axes: ['task_complexity', 'latency_tolerance', 'cost_ceiling', 'risk_tolerance', 'update_frequency']


## Walmart Use Case 1: Automated Returns Processing

**Business context:** Walmart processes approximately 1 million returns per day across all channels.
The current system requires a customer service associate to manually look up the purchase history,
check the return policy for the product category, determine eligibility, and process the refund.
The business wants to automate this for self-service kiosks and the Walmart app.

**Key facts:**
- Policy rules are documented and rarely change (quarterly updates)
- Input is structured: item ID, purchase date, reason code, customer ID
- 85% of cases follow one of six standard patterns
- Latency requirement: response within 3 seconds
- Cost sensitivity: millions of transactions per day
- A wrong decision (approving ineligible return) has direct financial impact

In [8]:
use_case_1 = {
    'name': 'Automated Returns Processing',
    'scores': {
        'task_complexity':   2,  # mostly deterministic policy lookup
        'latency_tolerance': 1,  # 3 second hard limit
        'cost_ceiling':      1,  # millions/day -- must be sub-cent
        'risk_tolerance':    1,  # wrong approval = direct financial loss
        'update_frequency':  2,  # policy changes quarterly
    },
    'constraints': {
        'latency_p95_sec': 3,
        'cost_ceiling_per_query_usd': 0.001,
        'daily_volume': 1_000_000,
    }
}

result_1 = recommend_architecture(use_case_1['scores'])
print(f'Use Case 1: {use_case_1["name"]}')
print(f'Total Score: {result_1["total_score"]} / 25')
print(f'Recommended Architecture: {result_1["architecture"]}')
print(f'Rationale: {result_1["rationale"]}')

Use Case 1: Automated Returns Processing
Total Score: 7 / 25
Recommended Architecture: Traditional Software
Rationale: Low complexity, strict latency, or low cost ceiling. A rule-based or deterministic pipeline is faster, cheaper, and more predictable.


## Walmart Use Case 2: Supplier Risk Intelligence

**Business context:** Walmart sources from over 100,000 suppliers globally. The procurement team
needs to continuously assess supplier risk across financial stability, geopolitical exposure,
regulatory compliance, and delivery performance. Currently, analysts manually read supplier reports,
news feeds, and financial filings to produce quarterly risk scorecards.

**Key facts:**
- Risk signals come from unstructured sources: news, filings, analyst reports, internal data
- The reasoning required varies significantly by supplier and risk type
- Analysts currently spend 3 hours per supplier per quarter
- Latency tolerance: results needed within 30 minutes of a news event
- Risk assessment logic changes as new risk categories emerge
- A missed risk signal has supply chain consequences, not immediate financial harm

In [9]:
use_case_2 = {
    'name': 'Supplier Risk Intelligence',
    'scores': {
        'task_complexity':   5,  # unstructured multi-source reasoning
        'latency_tolerance': 4,  # 30 min acceptable
        'cost_ceiling':      4,  # per-supplier cost, low volume
        'risk_tolerance':    3,  # missed signal is bad but not immediate
        'update_frequency':  4,  # risk categories evolve frequently
    },
    'constraints': {
        'latency_p95_sec': 1800,
        'cost_ceiling_per_query_usd': 2.00,
        'daily_volume': 500,
    }
}

result_2 = recommend_architecture(use_case_2['scores'])
print(f'Use Case 2: {use_case_2["name"]}')
print(f'Total Score: {result_2["total_score"]} / 25')
print(f'Recommended Architecture: {result_2["architecture"]}')
print(f'Rationale: {result_2["rationale"]}')

Use Case 2: Supplier Risk Intelligence
Total Score: 20 / 25
Recommended Architecture: Agent
Rationale: High complexity, dynamic branching, or rapidly changing requirements. An agent is justified -- but implement observability and cost controls from day one.


## Walmart Use Case 3: Store Performance Analytics Reporter

**Business context:** District managers oversee 15-20 Walmart stores each.
Every Monday morning, they manually compile a performance report from five internal systems
(sales data, inventory, staffing, shrinkage, customer satisfaction) and write a summary
with recommended actions for store managers.

**Key facts:**
- All five source systems have structured API access
- The report format is semi-standardised but the narrative varies by week
- District managers spend 2 hours on this every Monday
- Latency tolerance: report must be ready by 8am Monday
- The KPIs tracked are stable but the recommended actions depend on context
- Wrong recommendations are reviewed before action -- human in the loop exists

In [10]:
use_case_3 = {
    'name': 'Store Performance Analytics Reporter',
    'scores': {
        'task_complexity':   3,  # structured data pull + narrative generation
        'latency_tolerance': 5,  # overnight batch acceptable
        'cost_ceiling':      3,  # moderate -- weekly per district manager
        'risk_tolerance':    3,  # human reviews before action
        'update_frequency':  3,  # KPIs stable, narrative context varies
    },
    'constraints': {
        'latency_p95_sec': 3600,
        'cost_ceiling_per_query_usd': 0.50,
        'daily_volume': 50,
    }
}

result_3 = recommend_architecture(use_case_3['scores'])
print(f'Use Case 3: {use_case_3["name"]}')
print(f'Total Score: {result_3["total_score"]} / 25')
print(f'Recommended Architecture: {result_3["architecture"]}')
print(f'Rationale: {result_3["rationale"]}')

Use Case 3: Store Performance Analytics Reporter
Total Score: 17 / 25
Recommended Architecture: Workflow (Chain)
Rationale: Moderate complexity with known steps. A fixed-sequence workflow gives you LLM capability without agent overhead.


## Side-by-Side Architecture Recommendations

Review all three use cases together. Notice the pattern:
- High volume + strict latency + strict cost = traditional or workflow
- Low volume + complex reasoning + tolerant latency = agent
- The score is a starting point, not a final answer -- use it to structure the conversation

In [11]:
use_cases = [
    (use_case_1, result_1),
    (use_case_2, result_2),
    (use_case_3, result_3),
]

print(f"{'Use Case':<40} {'Score':>6} {'Recommended Architecture':<30}")
print('-' * 78)
for uc, res in use_cases:
    print(f'{uc["name"]:<40} {res["total_score"]:>6} {res["architecture"]:<30}')

print()
print('Key insight: Use Case 1 scores 7/25 -- a traditional policy engine is the right answer.')
print('Use Case 2 scores 20/25 -- the unstructured multi-source reasoning justifies an agent.')
print('Use Case 3 scores 17/25 -- a workflow with LLM narrative generation is sufficient.')

Use Case                                  Score Recommended Architecture      
------------------------------------------------------------------------------
Automated Returns Processing                  7 Traditional Software          
Supplier Risk Intelligence                   20 Agent                         
Store Performance Analytics Reporter         17 Workflow (Chain)              

Key insight: Use Case 1 scores 7/25 -- a traditional policy engine is the right answer.
Use Case 2 scores 20/25 -- the unstructured multi-source reasoning justifies an agent.
Use Case 3 scores 17/25 -- a workflow with LLM narrative generation is sufficient.


## LLM-Assisted Decision Justification

For ARB submission, the recommendation must be defensible in prose, not just a score.
The function below calls GPT-4o-mini to generate a one-paragraph ARB-ready justification
for each use case, grounded in the scored axes.

In [12]:
def generate_arb_justification(use_case: dict, recommendation: dict) -> str:
    axes_text = '\n'.join([
        f'  {axis}: {score}/5 -- {AXIS_DESCRIPTIONS[axis]}'
        for axis, score in use_case['scores'].items()
    ])
    prompt = f'''You are a senior AI architect writing a formal architecture decision note for a Walmart ARB review.

Use case: {use_case['name']}
Recommended architecture: {recommendation['architecture']}
Total decision score: {recommendation['total_score']} / 25

Axis scores:
{axes_text}

Write a single concise paragraph (4-6 sentences) that:
1. States the recommended architecture and why
2. Calls out the most influential axis scores
3. Names the primary risk of the alternative architectures
4. Is written in formal ARB language -- no hedging, no bullet points
'''
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()

print('Generating ARB justifications...')
for uc, res in use_cases:
    justification = generate_arb_justification(uc, res)
    uc['arb_justification'] = justification
    print(f'\n--- {uc["name"]} ---')
    print(justification)

Generating ARB justifications...

--- Automated Returns Processing ---
The recommended architecture for the Automated Returns Processing use case is Traditional Software, primarily due to its suitability for managing the task complexity, which scored 2 out of 5, indicating a moderate number of distinct decision points. The architecture is further justified by the low latency tolerance (1 out of 5) and cost ceiling (1 out of 5), reflecting the stringent requirements for timely processing and budget constraints. Additionally, the update frequency score of 2 out of 5 suggests that while changes may occur, they are not frequent enough to necessitate a more dynamic architecture. The primary risk associated with alternative architectures, such as cloud-native or microservices, lies in their potential inability to meet the stringent requirements for precision and response time, which are critical for effective returns processing. Therefore, the Traditional Software architecture is deemed the 

## Full Decision Matrix Builder

The complete decision matrix includes the use case context, axis scores, recommendation,
cost projections, and ARB justification. This is the format expected at architecture review.

In [13]:
def build_decision_matrix(use_case: dict, recommendation: dict) -> dict:
    constraints = use_case.get('constraints', {})
    daily_volume = constraints.get('daily_volume', 0)
    cost_per_query = constraints.get('cost_ceiling_per_query_usd', 0)
    monthly_cost_ceiling = daily_volume * cost_per_query * 30

    return {
        'use_case': use_case['name'],
        'axis_scores': use_case['scores'],
        'total_score': recommendation['total_score'],
        'recommended_architecture': recommendation['architecture'],
        'rationale': recommendation['rationale'],
        'arb_justification': use_case.get('arb_justification', ''),
        'constraints': {
            'latency_p95_sec': constraints.get('latency_p95_sec'),
            'cost_ceiling_per_query_usd': cost_per_query,
            'daily_volume': daily_volume,
            'monthly_cost_ceiling_usd': round(monthly_cost_ceiling, 2),
        },
        'alternatives_considered': [
            a for a in ['Traditional Software', 'Workflow (Chain)', 'Agent']
            if a != recommendation['architecture']
        ],
        'decision_status': 'DRAFT',
    }

matrices = []
for uc, res in use_cases:
    matrix = build_decision_matrix(uc, res)
    matrices.append(matrix)
    print(f'Built matrix for: {matrix["use_case"]}')
    print(f'  Architecture: {matrix["recommended_architecture"]}')
    print(f'  Monthly cost ceiling: ${matrix["constraints"]["monthly_cost_ceiling_usd"]:,.2f}')

Built matrix for: Automated Returns Processing
  Architecture: Traditional Software
  Monthly cost ceiling: $30,000.00
Built matrix for: Supplier Risk Intelligence
  Architecture: Agent
  Monthly cost ceiling: $30,000.00
Built matrix for: Store Performance Analytics Reporter
  Architecture: Workflow (Chain)
  Monthly cost ceiling: $750.00


## Lab Exercise: Design Your Own Matrix

Apply the five-axis model to a new Walmart use case of your choice.
Suggestions: demand forecasting assistant, HR policy Q&A, store layout optimiser,
vendor onboarding guide, or price matching automation.

Fill in the dictionary below and run the cell.

In [14]:
my_use_case = {
    'name': 'YOUR USE CASE NAME HERE',
    'scores': {
        'task_complexity':   3,  # replace with your score 1-5
        'latency_tolerance': 3,  # replace with your score 1-5
        'cost_ceiling':      3,  # replace with your score 1-5
        'risk_tolerance':    3,  # replace with your score 1-5
        'update_frequency':  3,  # replace with your score 1-5
    },
    'constraints': {
        'latency_p95_sec': 5,
        'cost_ceiling_per_query_usd': 0.01,
        'daily_volume': 1000,
    }
}

my_result = recommend_architecture(my_use_case['scores'])
my_justification = generate_arb_justification(my_use_case, my_result)
my_use_case['arb_justification'] = my_justification
my_matrix = build_decision_matrix(my_use_case, my_result)
matrices.append(my_matrix)

print(f'Your use case: {my_use_case["name"]}')
print(f'Score: {my_result["total_score"]} / 25')
print(f'Recommendation: {my_result["architecture"]}')
print()
print('ARB Justification:')
print(my_justification)

Your use case: YOUR USE CASE NAME HERE
Score: 15 / 25
Recommendation: Workflow (Chain)

ARB Justification:
The recommended architecture for the use case "YOUR USE CASE NAME HERE" is a Workflow (Chain) design, which effectively accommodates the task's complexity and the acceptable latency, cost, risk, and update frequency parameters. The most influential axis scores, each rated at 3 out of 5, indicate a moderate level of task complexity, latency tolerance, cost ceiling, risk tolerance, and update frequency, suggesting that a structured workflow can efficiently manage the distinct decision points while remaining responsive and cost-effective. Alternative architectures, such as a monolithic or event-driven approach, pose a significant risk of increased complexity and reduced flexibility, potentially leading to longer response times and higher costs in adapting to changing requirements. Therefore, the Workflow (Chain) architecture is deemed the most suitable choice for this use case, balan

## Validate and Export Decision Matrix

The validation function checks that every required field is complete and
no axis score is left at the default. This mirrors the completeness gate
used in the Module 5 ARB capstone.

In [15]:
def validate_decision_matrix(matrices: list) -> dict:
    results = []
    all_valid = True
    for m in matrices:
        issues = []
        if m['use_case'] in ('', 'YOUR USE CASE NAME HERE'):
            issues.append('Use case name is placeholder')
        for axis in AXES:
            score = m['axis_scores'].get(axis, 0)
            if not (1 <= score <= 5):
                issues.append(f'Axis {axis} score {score} is out of range 1-5')
        if not m.get('arb_justification'):
            issues.append('ARB justification is missing')
        status = 'PASS' if not issues else 'FAIL'
        if status == 'FAIL':
            all_valid = False
        results.append({'use_case': m['use_case'], 'status': status, 'issues': issues})
    return {'overall': 'PASS' if all_valid else 'FAIL', 'results': results}

validation = validate_decision_matrix(matrices)
print(f'Validation result: {validation["overall"]}')
for r in validation['results']:
    status_label = 'PASS' if r['status'] == 'PASS' else 'FAIL'
    print(f'  {status_label}  {r["use_case"]}')
    for issue in r['issues']:
        print(f'    - {issue}')

output = {
    'programme': 'Advanced Agentic AI -- Production Engineering',
    'track': 'India',
    'module': 'Module 1 -- Engineering Decisions for AI Systems',
    'validation': validation,
    'decision_matrices': matrices,
}

out_path = 'decision_matrix.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\nDecision matrix saved to {out_path}')
print(f'Total use cases documented: {len(matrices)}')

Validation result: FAIL
  PASS  Automated Returns Processing
  PASS  Supplier Risk Intelligence
  PASS  Store Performance Analytics Reporter
  FAIL  YOUR USE CASE NAME HERE
    - Use case name is placeholder

Decision matrix saved to decision_matrix.json
Total use cases documented: 4


## Summary

You have applied the five-axis decision model to three real Walmart use cases and produced
an ARB-ready decision matrix for each.

**Key takeaways:**

- The architecture decision is a function of volume, latency, cost, and risk -- not capability
- An agent is not always the best answer; over-engineering has real production costs
- The decision matrix forces you to quantify assumptions that are usually left implicit
- The ARB justification paragraph is what you will read aloud at the review board

**Your deliverable:** `decision_matrix.json`

**Next:** IN02 -- RAG vs Fine-tuning vs Prompting: Quantified Decision Framework and LoRA Lab